# 🏗️ Setup Lakehouse — CSV to Delta Tables

Reads the referential CSV files from the Lakehouse `Files/` folder
and writes them as typed Delta Parquet tables in `Tables/`.

**Tables created:**
- `dim_sites` — Factory sites (3 rows)
- `dim_zones` — Building zones per site (12 rows)
- `dim_sensors` — IoT sensors per zone (60 rows)

Run this notebook **once** after uploading the CSVs via `deploy_lakehouse.py`.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, BooleanType, DateType,
)

# Uses the default Lakehouse attached to this notebook.
# No hardcoded IDs — attach "LH_SensorReference" via the Lakehouse picker in the sidebar.
FILES = "Files"
TABLES = "Tables"

## 📋 dim_sites

In [ ]:
schema_sites = StructType([
    StructField("site_id", StringType()),
    StructField("site_name", StringType()),
    StructField("region", StringType()),
    StructField("country", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("is_active", BooleanType()),
])

df_sites = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema_sites)
    .load(f"{FILES}/dim_sites.csv")
)

df_sites.write.format("delta").mode("overwrite").save(f"{TABLES}/dim_sites")
print(f"✅ dim_sites — {df_sites.count()} rows written")
df_sites.show()

## 📋 dim_zones

In [ ]:
schema_zones = StructType([
    StructField("zone_id", StringType()),
    StructField("zone_name", StringType()),
    StructField("zone_type", StringType()),
    StructField("site_id", StringType()),
    StructField("floor", IntegerType()),
    StructField("area_sqm", DoubleType()),
    StructField("is_active", BooleanType()),
])

df_zones = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema_zones)
    .load(f"{FILES}/dim_zones.csv")
)

df_zones.write.format("delta").mode("overwrite").save(f"{TABLES}/dim_zones")
print(f"✅ dim_zones — {df_zones.count()} rows written")
df_zones.show()

## 📋 dim_sensors

In [ ]:
schema_sensors = StructType([
    StructField("sensor_id", StringType()),
    StructField("sensor_name", StringType()),
    StructField("sensor_type", StringType()),
    StructField("unit", StringType()),
    StructField("zone_id", StringType()),
    StructField("site_id", StringType()),
    StructField("min_normal", DoubleType()),
    StructField("max_normal", DoubleType()),
    StructField("min_critical", DoubleType()),
    StructField("max_critical", DoubleType()),
    StructField("install_date", DateType()),
    StructField("is_active", BooleanType()),
])

df_sensors = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema_sensors)
    .load(f"{FILES}/dim_sensors.csv")
)

df_sensors.write.format("delta").mode("overwrite").save(f"{TABLES}/dim_sensors")
print(f"✅ dim_sensors — {df_sensors.count()} rows written")
df_sensors.show(5)

## ✅ Verification

In [ ]:
for table in ["dim_sites", "dim_zones", "dim_sensors"]:
    df = spark.read.format("delta").load(f"{TABLES}/{table}")
    print(f"📊 {table}: {df.count()} rows, {len(df.columns)} columns")

print("\n✅ All Delta tables ready — Lakehouse setup complete.")